# Road Following - Late-fusion ResNet18 + ToF

Transfer-learn **ResNet18** on 224×224 images, then fuse two side ToF readings with a late-fusion head.

Images come from `1_data_collection_sensor.ipynb` (one folder up):

`road_following_A/apex_sensor/<x>_<y>_<tof_left>_<tof_right>_<uuid>.jpg`

(also `road_following_B/` if you used dataset B). Filenames store **raw ToF mm** and the click apex.

The model does **not** see raw mm. At train/infer time only:

- `0–300 mm` → linear `1 → 0` (`0 mm = 1`, `300 mm = 0`)
- anything else → `0` (ignored)

Apex `(x, y)` in the filename is the human label. The network predicts that apex.

### Architecture

Same ResNet18 as `1_data_collection_sensor.ipynb`, with `fc` widened to take the two ToF values.

```
image 224x224 -> ResNet18 (fc removed) -> 512
ToF x2        -> Linear(2, 16)         -> 16
concat(512, 16) -> Linear(528, output_dim) -> (x, y)
```

ToF is mapped `0–300 mm → 1–0` (closer is larger), and all other readings are `0`, before `Linear(2, 16)`. Raw mm stay in the filename.

### Dataset

Looks in `../road_following_A/apex_sensor/` (same folder collection writes), then B, then `../apex_sensor/` if you unzipped the zip next to the lab notebooks.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

# this notebook lives in scripts/; data + weights live with notebooks 0–3
ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.xy_dataset import XYDataset, find_dataset_roots

CATEGORIES = ['apex_sensor']

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# same folders as 1_data_collection_sensor.ipynb — NOT '.' (that looks for ./apex_sensor)
roots = find_dataset_roots(CATEGORIES)
print('cwd:', os.getcwd())
print('data roots:', roots or '(none)')
if not roots:
    raise FileNotFoundError(
        'No jpgs in road_following_A/apex_sensor, road_following_B/apex_sensor, '
        'or ./apex_sensor. Collect first with 1_data_collection_sensor.ipynb. '
        'Keep the lab notebooks in the JetRacer folder (parent of road_following_A).'
    )

dataset = XYDataset(roots, CATEGORIES, TRANSFORMS, random_hflip=True, return_sensors=True)

print('total:', len(dataset))
if len(dataset) == 0:
    raise FileNotFoundError('Folders exist but no parseable <x>_<y>_*.jpg files.')
for category in CATEGORIES:
    print(category, dataset.get_count(category))
for ann in dataset.annotations[:5]:
    print(os.path.basename(ann['image_path']),
          'xy=', (ann['x'], ann['y']),
          'tof=', (ann['tof_left'], ann['tof_right']))

Split train / test. ToF map is fixed (`0–300 mm` as `1→0`, else `0`) — no dataset mean/std.

In [ ]:
test_percent = 0.1
num_test = int(test_percent * len(dataset))
num_train = len(dataset) - num_test
if num_train < 1:
    raise ValueError('Need at least 1 image to train. Collect more in 1_data_collection_sensor.ipynb.')
if num_test == 0:
    train_dataset = dataset
    test_dataset = torch.utils.data.Subset(dataset, [])
else:
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [num_train, num_test])

BATCH_SIZE = 8

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=min(BATCH_SIZE, len(train_dataset)),
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print('train:', len(train_dataset), 'test:', len(test_dataset))

### Model

In [ ]:
from scripts.resnet_sensor_fusion import create_resnet18_sensor_fusion

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device', device)

output_dim = 2 * len(CATEGORIES)  # x, y coordinate for each category
model = create_resnet18_sensor_fusion(output_dim, pretrained=True)
model = model.to(device)

### Train

MSE on apex `(x, y)`. Each batch is `(image, sensors) -> (x, y)`.
The lowest test-loss weights are written to `best_steering_model_xy.pth`.

In [ ]:
NUM_EPOCHS = 70
BEST_MODEL_PATH = 'best_steering_model_xy.pth'
best_loss = 1e9
train_history = []
test_history = []

optimizer = torch.optim.Adam(model.parameters())
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)


def xy_mse_loss(outputs, category_idx, xy):
    loss = 0.0
    for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
        cat_idx = int(cat_idx)
        loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx + 2] - xy[batch_idx]) ** 2)
    return loss / len(category_idx)


for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for images, category_idx, xy, sensors in iter(train_loader):
        images = images.to(device)
        xy = xy.to(device)
        sensors = sensors.to(device)

        optimizer.zero_grad()
        outputs = model(images, sensors)
        loss = xy_mse_loss(outputs, category_idx, xy)
        train_loss += float(loss.detach())
        loss.backward()
        optimizer.step()
    train_loss /= max(len(train_loader), 1)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, category_idx, xy, sensors in iter(test_loader):
            images = images.to(device)
            xy = xy.to(device)
            sensors = sensors.to(device)
            outputs = model(images, sensors)
            loss = xy_mse_loss(outputs, category_idx, xy)
            test_loss += float(loss.detach())
    test_loss /= max(len(test_loader), 1)

    train_history.append(train_loss)
    test_history.append(test_loss)
    print('%d: train %f, test %f' % (epoch, train_loss, test_loss))
    if test_loss < best_loss:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_loss = test_loss
        print('saved', BEST_MODEL_PATH)

You can also save the latest weights under the name used by the interactive notebook. ToF scale is fixed (`0–300 mm` as `1→0`, else `0`) so it is not stored in the checkpoint.

In [ ]:
torch.save(model.state_dict(), 'road_following_model.pth')

### Visualize

Green = filename label (gt). Blue = model (pred).

Sliders start at the labelled ToF and stay raw mm, max 800. The net sees `1 - mm/300` if `0–300`, else `0`. Drag them to see how a closer sensor moves the prediction.

In [ ]:
import cv2
import ipywidgets
import PIL.Image
from IPython.display import display
from scripts.xy_dataset import normalize_tof_tensor

EVAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

if os.path.isfile(BEST_MODEL_PATH):
    try:
        ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
    state = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    print('loaded', BEST_MODEL_PATH)

dataset.refresh()
model.eval()


def jpeg(image_bgr):
    return bytes(cv2.imencode('.jpg', image_bgr)[1])


def predict_xy(image_bgr, tof_left, tof_right):
    rgb = PIL.Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    image = EVAL_TRANSFORMS(rgb).unsqueeze(0).to(device)
    sensors = normalize_tof_tensor(
        torch.tensor([[float(tof_left), float(tof_right)]])
    ).to(device)
    with torch.no_grad():
        out = model(image, sensors).detach().cpu().numpy().flatten()
    h, w = image_bgr.shape[:2]
    x = int(w * (float(out[0]) / 2.0 + 0.5))
    y = int(h * (float(out[1]) / 2.0 + 0.5))
    return x, y


def draw_point(image, xy, color, label):
    x, y = int(xy[0]), int(xy[1])
    cv2.circle(image, (x, y), 8, color, 2)
    cv2.putText(image, label, (x + 10, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)


def render(ann, tof_left, tof_right):
    image = cv2.imread(ann['image_path'], cv2.IMREAD_COLOR)
    if image is None:
        raise RuntimeError('failed to read %s' % ann['image_path'])
    tof_left = 0 if tof_left is None else int(tof_left)
    tof_right = 0 if tof_right is None else int(tof_right)
    px, py = predict_xy(image, tof_left, tof_right)
    vis = image.copy()
    draw_point(vis, (ann['x'], ann['y']), (0, 255, 0), 'gt')
    draw_point(vis, (px, py), (255, 0, 0), 'pred')
    cv2.putText(vis, 'L=%d R=%d' % (tof_left, tof_right),
                (6, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return vis, (px, py)

# test-set grid using each image's stored ToF
test_indices = list(getattr(test_dataset, 'indices', range(len(test_dataset))))
n_show = min(8, len(test_indices))
grid = []
for i in test_indices[:n_show]:
    vis, pred = render(dataset.annotations[i],
                             dataset.annotations[i]['tof_left'],
                             dataset.annotations[i]['tof_right'])
    grid.append(vis)
if grid:
    preview = cv2.hconcat(grid) if len(grid) <= 4 else cv2.vconcat([
        cv2.hconcat(grid[:4]),
        cv2.hconcat(grid[4:] + [np.zeros_like(grid[0])] * (8 - len(grid)))
    ])
    grid_widget = ipywidgets.Image(value=jpeg(preview), width=min(896, 224 * min(4, len(grid))))
    display(ipywidgets.VBox([
        ipywidgets.HTML('<b>Test predictions</b> (green=gt, blue=pred)'),
        grid_widget
    ]))

# interactive sliders
idx_slider = ipywidgets.IntSlider(description='image', min=0, max=max(len(dataset) - 1, 0), value=0)
tof_left_slider = ipywidgets.IntSlider(description='ToF left', min=20, max=800, value=400, step=5)
tof_right_slider = ipywidgets.IntSlider(description='ToF right', min=20, max=800, value=400, step=5)
live_widget = ipywidgets.Image(width=448, height=448)
status_widget = ipywidgets.HTML()


def set_sliders_from_image():
    ann = dataset.annotations[idx_slider.value]

    def clip(v):
        v = 400 if v is None else int(v)
        return max(20, min(800, v))

    tof_left_slider.value = clip(ann['tof_left'])
    tof_right_slider.value = clip(ann['tof_right'])


def update_view(*_):
    ann = dataset.annotations[idx_slider.value]
    vis, pred = render(ann, tof_left_slider.value, tof_right_slider.value)
    live_widget.value = jpeg(cv2.resize(vis, (448, 448), interpolation=cv2.INTER_NEAREST))
    status_widget.value = (
        'file xy=(%d,%d) &nbsp; pred=(%d,%d) &nbsp; slider ToF=(%d,%d) &nbsp; file ToF=(%s,%s)'
        % (ann['x'], ann['y'], pred[0], pred[1],
           tof_left_slider.value, tof_right_slider.value,
           ann['tof_left'], ann['tof_right'])
    )


def on_image_change(change):
    set_sliders_from_image()
    update_view()

idx_slider.observe(on_image_change, names='value')
tof_left_slider.observe(update_view, names='value')
tof_right_slider.observe(update_view, names='value')
set_sliders_from_image()
update_view()

display(ipywidgets.VBox([
    ipywidgets.HTML('<b>Interactive sensor test</b> — sliders start at the labelled ToF; drag them to see how closeness moves the blue prediction'),
    live_widget,
    idx_slider,
    tof_left_slider,
    tof_right_slider,
    status_widget
]))

### Next

`2_build_trt_model_sensor.ipynb` converts this two-input model. Deploy must apply the same ToF map: `0–300 mm → 1–0`, else `0`.